# Wan2.1 I2V 14B — API Server for Sahatuka Pipeline
**480P Diffusers | Portrait 9:16 for Reels**

Run all → copy **API URL** → paste in Phase 6.

In [ ]:
# @title 📦 1. Install
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate sentencepiece protobuf
!pip install -q fastapi uvicorn nest-asyncio imageio[ffmpeg] huggingface_hub

import os, torch, gc, uuid, time, threading, asyncio, subprocess, requests, random
from pathlib import Path
from huggingface_hub import login
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Enter your HuggingFace token (HF_TOKEN): ")
login(token=HF_TOKEN, add_to_git_credential=True)
print("✅ Logged in to HuggingFace")

In [ ]:
# @title 🤖 2. Load Model (480P Diffusers)
print("Loading Wan2.1 I2V 14B 480P... (~5 min)")

from diffusers import WanPipeline
from diffusers.utils import export_to_video
from PIL import Image

pipe = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers",
    torch_dtype=torch.float16, variant="fp16", device_map="balanced"
)
pipe.enable_model_cpu_offload()
print("✅ Model loaded!")

def save_video(frames, path, fps=16):
    export_to_video(frames, path, fps=fps)

In [ ]:
# @title 🌐 3. Start API Server
import nest_asyncio
nest_asyncio.apply()

!pkill -f cloudflared 2>/dev/null; sleep 1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI()
tasks = {}
API_URL = None

def start_tunnel():
    global API_URL
    proc = subprocess.Popen(
        ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8080"],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
    )
    for line in iter(proc.stderr.readline, b''):
        txt = line.decode().strip()
        if "trycloudflare.com" in txt:
            API_URL = "https://" + txt.split("https://")[-1].strip()
            print(f"\n✅  API URL: {API_URL}")
            print("⚠️  Copy this URL and paste it in Phase 6!\n")
            break

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}

@app.post("/generate")
async def generate(image: UploadFile = File(...), prompt: str = Form(...)):
    task_id = uuid.uuid4().hex[:8]
    img = Image.open(image.file).convert("RGB")
    img.save(f"/content/input_{task_id}.jpg")
    tasks[task_id] = {"state": "queued"}
    threading.Thread(target=_run, args=(task_id, prompt, img), daemon=True).start()
    return {"task_id": task_id}

def _run(task_id, prompt, img):
    tasks[task_id]["state"] = "running"
    try:
        img = img.resize((480, 832), Image.LANCZOS)
        video = pipe(
            prompt=prompt,
            negative_prompt="ugly, blurry, low quality, distorted",
            input_image=img,
            height=832, width=480,
            num_frames=49, num_inference_steps=20,
            seed=random.randint(0, 999999)
        )
        path = f"/content/output_{task_id}.mp4"
        save_video(video, path, fps=16)
        tasks[task_id]["state"] = "completed"
        tasks[task_id]["video_url"] = f"{API_URL}/video/{task_id}"
    except Exception as e:
        tasks[task_id]["state"] = "failed"
        tasks[task_id]["error"] = str(e)

@app.get("/status/{task_id}")
def status(task_id: str):
    return tasks.get(task_id, {"state": "unknown"})

@app.get("/video/{task_id}")
def video(task_id: str):
    if not API_URL: return {"error": "tunnel not ready"}
    path = f"/content/output_{task_id}.mp4"
    if os.path.exists(path):
        return FileResponse(path, media_type="video/mp4")
    return {"error": "not found"}

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(3)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8080, log_level="error")
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.create_task(server.serve())
print("✅ Server running!")

while API_URL is None:
    time.sleep(1)

---
Copy **API URL** → paste in Phase 6.

⚠️ Keep this Colab tab open.